In [8]:
import json
import os
import sys
import time
import traceback
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "agents").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from agents.ontology_qa_agent import build_ontology_qa_agent

In [9]:
DATA_PATH = PROJECT_ROOT / "data" / "test_questions_v1_0_valid_qa.json"
benchmark = json.loads(DATA_PATH.read_text(encoding="utf-8"))

print(f"Loaded {len(benchmark)} questions from {DATA_PATH}")
display(pd.Series(item["question_type"] for item in benchmark).value_counts().rename_axis("question_type").to_frame("total"))

Loaded 62 questions from /mnt/d/Dev/VDT2026-OntologyQA/data/test_questions_v1_0_valid_qa.json


,total
question_type,
entity,10
counting,10
attribute,9
boolean,8
list,7
schema,6
superlative,5
comparison,4
multi-hop,3


In [10]:
START_INDEX = 0
MAX_QUESTIONS = None  # None = chạy đến hết benchmark
MAX_ITERATIONS = 15
PRINT_STREAM_TO_NOTEBOOK = False  # True sẽ in toàn bộ stream; log file luôn được ghi

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = PROJECT_ROOT / "benchmark-logs" / run_id
QUESTION_LOG_DIR = RUN_DIR / "questions"
QUESTION_LOG_DIR.mkdir(parents=True, exist_ok=False)

selected_benchmark = benchmark[START_INDEX:]
if MAX_QUESTIONS is not None:
    selected_benchmark = selected_benchmark[:MAX_QUESTIONS]

run_config = {
    "run_id": run_id,
    "dataset": str(DATA_PATH),
    "start_index": START_INDEX,
    "max_questions": MAX_QUESTIONS,
    "question_count": len(selected_benchmark),
    "max_iterations": MAX_ITERATIONS,
    "model": os.environ.get("OPENROUTER_MODEL"),
}
(RUN_DIR / "run_config.json").write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")

graph = build_ontology_qa_agent(max_iterations=MAX_ITERATIONS)
print(f"Run directory: {RUN_DIR}")
print(f"Questions to run: {len(selected_benchmark)}")

Run directory: /mnt/d/Dev/VDT2026-OntologyQA/benchmark-logs/20260625_140537
Questions to run: 62


In [11]:
def json_text(value):
    return json.dumps(value, ensure_ascii=False, indent=2, default=str)


def extract_token_usage(message):
    metadata = getattr(message, "response_metadata", {}) or {}
    usage = metadata.get("token_usage", {}) or {}

    input_tokens = int(usage.get("prompt_tokens") or usage.get("input_tokens") or 0)
    output_tokens = int(usage.get("completion_tokens") or usage.get("output_tokens") or 0)
    total_tokens = int(usage.get("total_tokens") or (input_tokens + output_tokens))

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
    }


def format_stream_event(event):
    sections = []
    for node_name, update in event.items():
        lines = [f"{'=' * 20} {node_name} {'=' * 20}"]
        if node_name == "initialize":
            lines.append(f"initialized keys: {sorted(update)}")
        elif node_name == "agent":
            message = update["messages"][-1]
            lines.append(f"content: {message.content}")
            lines.append(f"tool_calls: {json_text(message.tool_calls)}")

            usage = extract_token_usage(message)
            if usage["total_tokens"] > 0:
                lines.append(
                    "token_usage: "
                    f"input={usage['input_tokens']}, "
                    f"output={usage['output_tokens']}, "
                    f"total={usage['total_tokens']}"
                )
        elif node_name == "tools":
            for message in update.get("messages", []):
                lines.append(f"{message.name}: {message.content}")
        elif node_name == "post_tool":
            lines.append("generated_sparqls:")
            lines.extend(repr(query) for query in update.get("generated_sparqls", []))
            lines.append(f"executions: {json_text(update.get('executions', []))}")
        elif node_name == "retry":
            lines.append(str(update["messages"][-1].content))
        else:
            lines.append(json_text(update))
        sections.append("\n".join(lines))
    return "\n\n".join(sections) + "\n"


def save_checkpoint(records):
    (RUN_DIR / "results.json").write_text(json.dumps(records, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    csv_records = []
    for record in records:
        csv_record = {**record, "result": json.dumps(record.get("result"), ensure_ascii=False, default=str)}
        csv_records.append(csv_record)
    pd.DataFrame(csv_records).to_csv(RUN_DIR / "results.csv", index=False, encoding="utf-8-sig")

In [12]:
records = []
total_to_run = len(selected_benchmark)

for position, item in enumerate(selected_benchmark, start=1):
    question_id = item["id"]
    question_type = item["question_type"]
    log_path = QUESTION_LOG_DIR / f"{int(question_id):03d}_{question_type}.log"
    started_at = time.perf_counter()
    final_update = None
    error = None
    input_tokens = 0
    output_tokens = 0

    header = {
        "id": question_id,
        "question_type": question_type,
        "question": item["question"],
        "options": item["options"],
        "correct_option_id": item["correct_option_id"],
        "gold_answer": item.get("gold_answer"),
    }

    print(f"[{position:02d}/{total_to_run:02d}] id={question_id} type={question_type}: {item['question']}")
    with log_path.open("w", encoding="utf-8") as log_file:
        log_file.write("BENCHMARK ITEM\n")
        log_file.write(json_text(header) + "\n\n")
        try:
            graph_input = {"question": item["question"], "options": item["options"]}
            for event in graph.stream(graph_input, stream_mode="updates"):
                event_text = format_stream_event(event)
                log_file.write(event_text + "\n")
                log_file.flush()
                if PRINT_STREAM_TO_NOTEBOOK:
                    print(event_text)

                if "agent" in event:
                    message = event["agent"]["messages"][-1]
                    usage = extract_token_usage(message)
                    input_tokens += usage["input_tokens"]
                    output_tokens += usage["output_tokens"]

                if "finalize" in event:
                    final_update = event["finalize"]
            if final_update is None:
                raise RuntimeError("Graph stream ended without a finalize update.")
        except Exception as exc:
            error = f"{type(exc).__name__}: {exc}"
            log_file.write("\nEXCEPTION\n")
            log_file.write(traceback.format_exc())

        elapsed_seconds = round(time.perf_counter() - started_at, 3)
        selected_option_id = final_update.get("selected_option_id") if final_update else None
        answer = final_update.get("answer") if final_update else None
        result = final_update.get("result") if final_update else None
        is_correct = error is None and selected_option_id == item["correct_option_id"]
        status = "error" if error else ("correct" if is_correct else "incorrect")

        total_tokens = input_tokens + output_tokens
        tokens_per_second = round(total_tokens / elapsed_seconds, 2) if elapsed_seconds > 0 else None

        record = {
            **header,
            "selected_option_id": selected_option_id,
            "answer": answer,
            "result": result,
            "is_correct": is_correct,
            "status": status,
            "error": error,
            "elapsed_seconds": elapsed_seconds,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
            "tokens_per_second": tokens_per_second,
            "log_file": str(log_path.relative_to(PROJECT_ROOT)),
        }
        records.append(record)
        log_file.write("\nBENCHMARK RESULT\n")
        log_file.write(json_text(record) + "\n")

    save_checkpoint(records)
    print(
        f"    {status.upper()} | predicted={selected_option_id} expected={item['correct_option_id']} "
        f"| {elapsed_seconds:.1f}s | tokens={total_tokens} (in={input_tokens}, out={output_tokens}) "
        f"| {log_path.name}"
    )

[01/62] id=1 type=entity: Thành phố Hồ Chí Minh thuộc quốc gia nào?
    CORRECT | predicted=3 expected=3 | 12.3s | tokens=28355 (in=28090, out=265) | 001_entity.log
[02/62] id=2 type=entity: Ai là người thành lập Thành phố Hồ Chí Minh
    CORRECT | predicted=4 expected=4 | 13.7s | tokens=39108 (in=38755, out=353) | 002_entity.log
[03/62] id=3 type=entity: Nguyễn Hữu Cảnh mất tại tỉnh nào
    CORRECT | predicted=5 expected=5 | 13.4s | tokens=29410 (in=29027, out=383) | 003_entity.log
[04/62] id=4 type=entity: Ai là vợ của Barack Obama?
    CORRECT | predicted=4 expected=4 | 12.9s | tokens=30332 (in=30088, out=244) | 004_entity.log
[05/62] id=5 type=entity: Tàu INS Vikrant (2013) được xây dựng ở đâu
    CORRECT | predicted=3 expected=3 | 37.9s | tokens=139731 (in=138875, out=856) | 005_entity.log
[06/62] id=6 type=entity: Máy bay Mil Mi-8 được sản xuất bởi ai
    CORRECT | predicted=1 expected=1 | 9.8s | tokens=20228 (in=20084, out=144) | 006_entity.log
[07/62] id=7 type=entity: Tiền thâ

In [13]:
results_df = pd.DataFrame(records)

summary = results_df.groupby("question_type", sort=True).agg(
    total=("id", "count"),
    correct=("is_correct", "sum"),
    errors=("status", lambda values: (values == "error").sum()),
    avg_elapsed_seconds=("elapsed_seconds", "mean"),
    avg_input_tokens=("input_tokens", "mean"),
    avg_output_tokens=("output_tokens", "mean"),
    avg_total_tokens=("total_tokens", "mean"),
)
summary["incorrect"] = summary["total"] - summary["correct"] - summary["errors"]
summary["accuracy"] = (summary["correct"] / summary["total"]).round(4)
summary["avg_elapsed_seconds"] = summary["avg_elapsed_seconds"].round(3)
summary["avg_input_tokens"] = summary["avg_input_tokens"].round(2)
summary["avg_output_tokens"] = summary["avg_output_tokens"].round(2)
summary["avg_total_tokens"] = summary["avg_total_tokens"].round(2)
summary = summary[[
    "total",
    "correct",
    "incorrect",
    "errors",
    "accuracy",
    "avg_elapsed_seconds",
    "avg_input_tokens",
    "avg_output_tokens",
    "avg_total_tokens",
]]

total_row = pd.DataFrame(
    {
        "total": [int(summary["total"].sum())],
        "correct": [int(summary["correct"].sum())],
        "incorrect": [int(summary["incorrect"].sum())],
        "errors": [int(summary["errors"].sum())],
        "accuracy": [round(float(summary["correct"].sum() / summary["total"].sum()), 4)],
        "avg_elapsed_seconds": [round(float(results_df["elapsed_seconds"].mean()), 3)],
        "avg_input_tokens": [round(float(results_df["input_tokens"].mean()), 2)],
        "avg_output_tokens": [round(float(results_df["output_tokens"].mean()), 2)],
        "avg_total_tokens": [round(float(results_df["total_tokens"].mean()), 2)],
    },
    index=["ALL"],
)
summary_with_total = pd.concat([summary, total_row])
summary_with_total.to_csv(RUN_DIR / "summary_by_type.csv", encoding="utf-8-sig")
(RUN_DIR / "summary_by_type.json").write_text(summary_with_total.reset_index(names="question_type").to_json(orient="records", force_ascii=False, indent=2), encoding="utf-8")

display(summary_with_total)
print(f"Detailed results: {RUN_DIR / 'results.csv'}")
print(f"Summary by type: {RUN_DIR / 'summary_by_type.csv'}")
print(f"Per-question stream logs: {QUESTION_LOG_DIR}")

,total,correct,incorrect,errors,accuracy,avg_elapsed_seconds,avg_input_tokens,avg_output_tokens,avg_total_tokens
attribute,9,9,0,0,1.0000,14.161,25570.00,244.33,25814.33
boolean,8,7,0,1,0.8750,44.186,46127.88,472.25,46600.12
comparison,4,4,0,0,1.0000,23.226,46966.75,504.50,47471.25
counting,10,7,3,0,0.7000,37.478,97110.50,645.90,97756.40
entity,10,9,1,0,0.9000,16.089,43893.40,363.30,44256.70
list,7,5,2,0,0.7143,74.734,62434.57,3023.14,65457.71
multi-hop,3,2,1,0,0.6667,63.347,105068.33,701.33,105769.67
schema,6,4,2,0,0.6667,16.553,31757.17,270.67,32027.83
superlative,5,3,2,0,0.6000,26.738,66566.00,644.20,67210.20
ALL,62,50,11,1,0.8065,33.156,56010.95,745.13,56756.08


Detailed results: /mnt/d/Dev/VDT2026-OntologyQA/benchmark-logs/20260625_140537/results.csv
Summary by type: /mnt/d/Dev/VDT2026-OntologyQA/benchmark-logs/20260625_140537/summary_by_type.csv
Per-question stream logs: /mnt/d/Dev/VDT2026-OntologyQA/benchmark-logs/20260625_140537/questions


In [14]:
debug_columns = ["id", "question_type", "question", "correct_option_id", "selected_option_id", "status", "log_file"]
display(results_df.loc[results_df["status"] != "correct", debug_columns])

,id,question_type,question,correct_option_id,selected_option_id,status,log_file
9,10,entity,Victor Hugo có tác phẩm nào là truyện tranh k...,1,2.0,incorrect,benchmark-logs/20260625_140537/questions/010_e...
15,16,counting,Có bao nhiêu chiếc máy bay có nguồn gốc từ Liê...,2,NaN,incorrect,benchmark-logs/20260625_140537/questions/016_c...
16,17,counting,Số lượng nghệ sĩ có ngày sinh từ 1986-01-01 t...,5,NaN,incorrect,benchmark-logs/20260625_140537/questions/017_c...
18,19,counting,Nêu tổng số tác phẩm của Victor Hugo?,2,NaN,incorrect,benchmark-logs/20260625_140537/questions/019_c...
25,26,list,Danh sách những con tàu mà hãng Boeing đóng?,5,4.0,incorrect,benchmark-logs/20260625_140537/questions/026_l...
26,27,list,Đâu là các cặp vợ chồng cùng làm nghề thủy thủ...,2,NaN,incorrect,benchmark-logs/20260625_140537/questions/027_l...
31,35,boolean,Tàu ngầm Việt Nam Bà rịa Vũng tàu có cảng đăng...,2,NaN,error,benchmark-logs/20260625_140537/questions/035_b...
37,43,multi-hop,Kể tên vị trí của những nhà sản xuất máy bay đ...,4,3.0,incorrect,benchmark-logs/20260625_140537/questions/043_m...
45,64,schema,Đâu là một loại hình công trình/hay tác phẩm p...,1,NaN,incorrect,benchmark-logs/20260625_140537/questions/064_s...
47,66,schema,Thuộc tính trần bay của một máy bay có tên đún...,1,NaN,incorrect,benchmark-logs/20260625_140537/questions/066_s...
